# 05 — Key Vault mass access hunt

For the **Defender for Key Vault** scenarios (S-KV-01/02/03/04/08).
Requires diagnostic settings on the vault sending `AuditEvent` to LAW.


In [ ]:
"""Shared setup — run me first.
Connects to Log Analytics via DefaultAzureCredential (uses az login, env vars, or managed identity).
"""
import os
import pandas as pd
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

WORKSPACE_ID = os.environ.get("LAW_WORKSPACE_ID")  # GUID of the Log Analytics workspace
assert WORKSPACE_ID, "Set LAW_WORKSPACE_ID env var (workspace customerId GUID, not full ARM id)"

client = LogsQueryClient(DefaultAzureCredential())

def kql(query: str, hours: int = 24) -> pd.DataFrame:
    """Run a KQL query against the workspace and return the first table as a DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        print("WARNING: partial result -", resp.partial_error)
        tables = resp.partial_data
    elif resp.status == LogsQueryStatus.SUCCESS:
        tables = resp.tables
    else:
        raise RuntimeError(f"Query failed: {resp}")
    if not tables:
        return pd.DataFrame()
    t = tables[0]
    return pd.DataFrame(t.rows, columns=t.columns)

pd.set_option("display.max_colwidth", 120)
print("Workspace:", WORKSPACE_ID)


## Identities reading many secrets in a short window

In [ ]:
df = kql("""
KeyVaultData
| where TimeGenerated > ago(24h)
| where OperationName in ("SecretGet","SecretList","SecretListVersions")
| where ResultType == "Success"
| summarize SecretsAccessed=dcount(id_s),
            Ops=count(),
            Vaults=dcount(Resource)
          by identity_claim_unique_name_s, CallerIPAddress, bin(TimeGenerated, 15m)
| where SecretsAccessed > 10
| order by SecretsAccessed desc
""")
df


## Denied access spikes

In [ ]:
df = kql("""
KeyVaultData
| where TimeGenerated > ago(24h)
| where ResultType != "Success"
| summarize Denied=count(), Ops=make_set(OperationName, 5)
          by identity_claim_unique_name_s, CallerIPAddress, Resource
| where Denied > 20
| order by Denied desc
""")
df


## Policy changes followed by enumeration

In [ ]:
df = kql("""
let policyChanges = KeyVaultData
    | where TimeGenerated > ago(24h)
    | where OperationName in ("VaultPut","VaultPatch","VaultAccessPolicyChangeEvent");
let bulkReads = KeyVaultData
    | where TimeGenerated > ago(24h)
    | where OperationName in ("SecretGet","SecretList")
    | summarize SecretsRead=count() by identity_claim_unique_name_s, Vault=Resource, bin(TimeGenerated, 15m)
    | where SecretsRead > 5;
policyChanges
| join kind=inner bulkReads on $left.identity_claim_unique_name_s == $right.identity_claim_unique_name_s
                              and $left.Resource == $right.Vault
| project TimeGenerated, identity_claim_unique_name_s, Vault, PolicyOp=OperationName, SecretsRead
""")
df


## Deleted + purged secrets

In [ ]:
df = kql("""
KeyVaultData
| where TimeGenerated > ago(24h)
| where OperationName in ("SecretDelete","SecretPurge")
| summarize Count=count(), Ops=make_set(OperationName) by identity_claim_unique_name_s, Resource
| where Count > 1
""")
df
